# 🔬 Ví dụ giải thích SIÊU DỄ: Claude QUYẾT ĐỊNH gọi công cụ nào?

> File bổ sung #2 — đi cặp với `example_agentic_loop_explained.ipynb`.
>
> - File trước trả lời: *"Vòng lặp chạy thế nào?"* (cơ chế)
> - File này trả lời: **"Claude DỰA VÀO ĐÂU để chọn đúng công cụ?"** (bộ não ra quyết định)

Câu hỏi cốt lõi: bạn đưa Claude 3 công cụ, nó làm sao biết câu này dùng cái nào? Có khi nào nó gọi sai, gọi thừa, hay quên gọi không?

Ta sẽ làm **thí nghiệm** để thấy tận mắt: **`description` của công cụ chính là "bộ não" điều khiển lựa chọn.**


## 1. Chuẩn bị: hàm "đọc quyết định" của Claude

Hàm dưới gọi Claude kèm danh sách công cụ, nhưng **KHÔNG thực thi** gì cả — chỉ **in ra Claude ĐÃ CHỌN gì**. Ta chỉ quan tâm phần "ra quyết định".


In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

def xem_quyet_dinh(cau_hoi, tools, tool_choice=None):
    kwargs = dict(model=MODEL, max_tokens=512, tools=tools,
                  messages=[{"role": "user", "content": cau_hoi}])
    if tool_choice:
        kwargs["tool_choice"] = tool_choice

    res = client.messages.create(**kwargs)
    print(f"❓ Hỏi: {cau_hoi}")
    print(f"   stop_reason = {res.stop_reason}")

    goi_cong_cu = [b for b in res.content if b.type == "tool_use"]
    if not goi_cong_cu:
        # Claude trả lời thẳng, không gọi công cụ
        txt = next((b.text for b in res.content if b.type == "text"), "")
        print(f"   → KHÔNG gọi công cụ. Claude tự trả lời: {txt[:80]}")
    else:
        for b in goi_cong_cu:
            print(f"   → CHỌN: {b.name}({b.input})")
    print()

## 2. Thí nghiệm A — Description rõ ràng → Claude định tuyến ĐÚNG

3 công cụ, mỗi cái có `description` nói rõ **khi nào dùng**. Xem Claude tự chọn đúng cho từng câu hỏi.


In [ ]:
tools_ro = [
    {
        "name": "calculator",
        "description": "Thực hiện phép tính số học. Dùng khi cần tính toán con số.",
        "input_schema": {"type": "object",
            "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
    },
    {
        "name": "get_weather",
        "description": "Lấy thời tiết hiện tại của một thành phố. Dùng khi hỏi về thời tiết/nhiệt độ.",
        "input_schema": {"type": "object",
            "properties": {"city": {"type": "string"}}, "required": ["city"]},
    },
    {
        "name": "translate",
        "description": "Dịch một câu sang ngôn ngữ khác. Dùng khi cần dịch thuật.",
        "input_schema": {"type": "object",
            "properties": {"text": {"type": "string"}, "to_lang": {"type": "string"}},
            "required": ["text", "to_lang"]},
    },
]

xem_quyet_dinh("123 nhân 45 bằng mấy?", tools_ro)
xem_quyet_dinh("Hôm nay Đà Nẵng thời tiết sao?", tools_ro)
xem_quyet_dinh("Dịch 'xin chào' sang tiếng Nhật.", tools_ro)

👀 **Quan sát:** mỗi câu hỏi → Claude chọn **đúng 1 công cụ** tương ứng. Nó đọc `description` để khớp ý định của bạn với công cụ phù hợp.


## 3. Thí nghiệm B — Description MƠ HỒ → Claude lúng túng

Vẫn 1 công cụ tính toán, nhưng `description` viết tệ ("Một công cụ hữu ích"). Xem Claude có còn chọn đúng không.


In [ ]:
tools_mo_ho = [
    {
        "name": "calculator",
        "description": "Một công cụ hữu ích.",   # ❌ mô tả vô nghĩa
        "input_schema": {"type": "object",
            "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
    },
]

print("=== Mô tả MƠ HỒ ===")
xem_quyet_dinh("567 cộng 433 bằng bao nhiêu?", tools_mo_ho)

print("=== Mô tả RÕ (để so sánh) ===")
xem_quyet_dinh("567 cộng 433 bằng bao nhiêu?", tools_ro)

👀 **Bài học #1:** `description` **chính là bộ não** điều khiển lựa chọn. Mô tả tệ → Claude có thể không gọi, gọi sai thời điểm, hoặc tự tính (và tính sai). **Đầu tư viết `description` thật rõ "công cụ này làm gì, khi nào dùng".**


## 4. Thí nghiệm C — Câu hỏi KHÔNG cần công cụ → Claude tự trả lời

Quan trọng: có công cụ **không** nghĩa là Claude **luôn** gọi. Nếu câu hỏi không liên quan, nó trả lời thẳng (`stop_reason = end_turn`).


In [ ]:
xem_quyet_dinh("Thủ đô nước Pháp là gì?", tools_ro)     # kiến thức sẵn có
xem_quyet_dinh("Kể một câu chuyện cười ngắn.", tools_ro)   # không cần công cụ

👀 **Bài học #2:** Claude **tự phân biệt** khi nào cần công cụ, khi nào không. `tool_choice` mặc định là `"auto"` (Claude tự quyết). Nó không "cuồng" gọi công cụ bừa.


## 5. Thí nghiệm D — Một câu cần NHIỀU công cụ → gọi song song

Nếu câu hỏi cần nhiều việc độc lập, Claude có thể trả về **nhiều khối `tool_use` cùng lúc** (parallel tool use).


In [ ]:
xem_quyet_dinh(
    "Cho mình biết thời tiết Hà Nội và đồng thời tính 15% của 800.",
    tools_ro,
)

👀 **Bài học #3:** một lượt trả lời của Claude có thể chứa **nhiều** lời gọi công cụ. Khi xử lý, nhớ **duyệt hết** các khối `tool_use` và trả về **tất cả** `tool_result` trong một message `user` (xem file vòng lặp).


## 6. Thí nghiệm E — ÉP buộc lựa chọn với `tool_choice`

Bạn có thể **ghi đè** quyết định của Claude:

| `tool_choice` | Ý nghĩa |
|---|---|
| `{"type": "auto"}` | Claude tự quyết (mặc định) |
| `{"type": "any"}` | BẮT BUỘC gọi ít nhất 1 công cụ |
| `{"type": "tool", "name": "X"}` | BẮT BUỘC gọi đúng công cụ X |
| `{"type": "none"}` | CẤM gọi công cụ |


In [ ]:
# Cùng 1 câu hỏi "vu vơ" nhưng ÉP phải gọi công cụ
print("=== auto (tự quyết) ===")
xem_quyet_dinh("Chào buổi sáng!", tools_ro, tool_choice={"type": "auto"})

print("=== any (ép phải gọi 1 công cụ nào đó) ===")
xem_quyet_dinh("Chào buổi sáng!", tools_ro, tool_choice={"type": "any"})

print("=== ép đúng calculator ===")
xem_quyet_dinh("Tính giúp gì đó đi", tools_ro, tool_choice={"type": "tool", "name": "calculator"})

👀 **Bài học #4:** dùng `tool_choice` khi bạn cần **kiểm soát**: ví dụ bắt buộc trích xuất dữ liệu qua một công cụ cố định, hoặc tạm cấm công cụ trong một số bước.


## 7. Tổng kết logic chọn công cụ

```
Claude nhận: câu hỏi  +  danh sách tools (mỗi tool có description)
        │
        ▼
   So khớp Ý ĐỊNH trong câu hỏi  ⟷  DESCRIPTION của từng tool
        │
        ├─ Khớp 1 tool      → gọi 1 tool
        ├─ Khớp nhiều tool  → gọi song song nhiều tool
        ├─ Không khớp tool  → tự trả lời (end_turn)
        └─ Bị tool_choice ép → tuân theo ép buộc
```

### 4 bài học rút ra
1. **`description` = bộ não lựa chọn.** Viết rõ "làm gì, khi nào dùng".
2. Có tool ≠ luôn gọi tool. Claude tự phân biệt (mặc định `auto`).
3. Một lượt có thể gọi **nhiều** tool cùng lúc → xử lý hết.
4. Cần kiểm soát → dùng **`tool_choice`** (auto / any / tool / none).


---
## ✅ Tóm tắt 1 dòng

> Claude chọn công cụ bằng cách **so khớp ý định trong câu hỏi với `description` của từng tool**. Muốn nó chọn đúng → viết description rõ; muốn ép → dùng `tool_choice`.
